<a href="https://colab.research.google.com/github/nexageapps/LLM/blob/main/02_Intermediate/L16_Transfer_Learning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# L16: Transfer Learning - Reusing Pre-trained Models

**Author:** Karthik Arjun  
**LinkedIn:** [Karthik Arjun](https://www.linkedin.com/in/karthik-arjun-a5b4a258/)  
**Level:** Intermediate  
**Lesson:** 16 of 30

---

## Learning Objectives

By the end of this lesson, you will:
- Understand transfer learning and why it works for NLP
- Differentiate feature extraction from full fine-tuning
- Freeze and unfreeze layers for different strategies
- Add task-specific heads on top of pre-trained encoders
- Apply transfer learning to a text classification task
- Explore domain adaptation concepts

---

## Concept: What is Transfer Learning?

**Transfer learning** reuses a model trained on one task (e.g. language modeling) for another task (e.g. sentiment classification) with limited task-specific data.

- **Pre-trained model**: Already learned general language representations.
- **Feature extraction**: Use the model as a fixed encoder; train only a new head (fast, fewer parameters).
- **Fine-tuning**: Update some or all pre-trained weights on your task (usually better accuracy, more compute).
- **Domain adaptation**: Adapt a general model to a specific domain (e.g. medical, legal) via continued pre-training or fine-tuning.

---

In [ ]:
# Step 1: Install and Import

!pip install transformers torch accelerate datasets -q

import torch
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from datasets import load_dataset
import warnings
warnings.filterwarnings('ignore')

print("Libraries loaded.")
print(f"Device: {'cuda' if torch.cuda.is_available() else 'cpu'}")

## Part 1: Loading a Pre-trained Model and Inspecting Layers

We use a small encoder (DistilBERT) suitable for classification. You can freeze its body and train only the classification head (feature extraction) or unfreeze it (fine-tuning).

---

In [ ]:
# Step 2: Load Pre-trained Model and Tokenizer

model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

print(f"Model: {model_name}")
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")
print("\nLayer groups:")
for name, _ in model.named_parameters():
    print(f"  {name}")
    break  # show first only; remove break to list all

## Part 2: Feature Extraction (Freeze Backbone)

Freeze all parameters of the pre-trained transformer and train only the classification head. Faster and uses less memory; good when you have little data or want to avoid overwriting pre-trained knowledge.

---

In [ ]:
# Step 3: Freeze Backbone for Feature Extraction

def set_trainable_layers(model, trainable_layer_names=None, freeze_all_backbone=True):
    """Freeze backbone; optionally make only classifier trainable."""
    for param in model.parameters():
        param.requires_grad = False
    if freeze_all_backbone:
        for name, param in model.named_parameters():
            if "classifier" in name or "pre_classifier" in name:
                param.requires_grad = True
    return model

model_fe = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)
model_fe = set_trainable_layers(model_fe, freeze_all_backbone=True)

trainable = sum(p.numel() for p in model_fe.parameters() if p.requires_grad)
total = sum(p.numel() for p in model_fe.parameters())
print(f"Trainable: {trainable:,} / {total:,} ({100*trainable/total:.1f}%)")

## Part 3: Prepare a Small Dataset

We use a tiny subset of a sentiment dataset so the lesson runs quickly. You can switch to full datasets (e.g. IMDB) for real experiments.

---

In [ ]:
# Step 4: Load and Prepare Data

def get_sample_dataset(split="train", sample_size=500, text_key="text", label_key="label"):
    ds = load_dataset("imdb", split=split, trust_remote_code=True)
    ds = ds.select(range(min(sample_size, len(ds))))
    return ds

def tokenize_fn(examples):
    return tokenizer(examples["text"], truncation=True, max_length=128, padding="max_length")

train_ds = get_sample_dataset("train", 400)
eval_ds = get_sample_dataset("test", 100)

train_ds = train_ds.map(tokenize_fn, batched=True, remove_columns=["text"])
eval_ds = eval_ds.map(tokenize_fn, batched=True, remove_columns=["text"])
train_ds.set_format("torch")
eval_ds.set_format("torch")

print(f"Train size: {len(train_ds)}, Eval size: {len(eval_ds)}")

## Part 4: Train with Feature Extraction (Frozen Backbone)

Training only the head for a few epochs. This is fast and often gives a good baseline.

---

In [ ]:
# Step 5: Train Feature-Extraction Model

training_args = TrainingArguments(
    output_dir="./out_transfer_l16",
    num_train_epochs=2,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    logging_steps=20,
    report_to="none",
)

trainer_fe = Trainer(
    model=model_fe,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
)

trainer_fe.train()
eval_fe = trainer_fe.evaluate()
print("\nFeature extraction eval:", eval_fe)

## Part 5: Full Fine-tuning (All Layers Trainable)

Unfreeze the entire model and train with a small learning rate so we do not destroy pre-trained representations. Typically gives better accuracy than feature extraction when you have enough data.

---

In [ ]:
# Step 6: Full Fine-tuning (Unfreeze All)

model_ft = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)
for param in model_ft.parameters():
    param.requires_grad = True

training_args_ft = TrainingArguments(
    output_dir="./out_finetune_l16",
    num_train_epochs=2,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    evaluation_strategy="epoch",
    learning_rate=2e-5,
    save_strategy="epoch",
    logging_steps=20,
    report_to="none",
)

trainer_ft = Trainer(
    model=model_ft,
    args=training_args_ft,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
)

trainer_ft.train()
eval_ft = trainer_ft.evaluate()
print("\nFull fine-tuning eval:", eval_ft)
print("\nCompare: feature extraction vs full fine-tuning (same data, 2 epochs).")

## Part 6: Domain Adaptation Idea

**Domain adaptation** means making a model work better in a specific domain (e.g. medical, legal, code). Options:
1. **Continued pre-training** on in-domain text (MLM or CLM), then fine-tune on the task.
2. **Fine-tune on in-domain labeled data** for your task.
3. **Vocabulary extension** if the domain has many new tokens.

Here we only illustrate the pipeline: you would replace the dataset with your domain data.

---

In [ ]:
# Step 7: Domain-Specific Fine-tuning (Same API, Different Data)

# Example: same code, but you would load domain-specific data:
# domain_train = load_dataset("your_org/medical_ner", split="train")
# Then tokenize and train as above with model_ft.

print("Domain adaptation = same transfer-learning pipeline with domain data.")
print("1. Option A: Continue pre-training on domain text (MLM), then fine-tune.")
print("2. Option B: Fine-tune directly on domain-labeled task data.")
print("3. Option C: Extend tokenizer/vocab if needed, then fine-tune.")

## Exercises

### Exercise 1: Freeze Only First N Layers
Modify the model so only the last 2 transformer layers and the classifier are trainable. Compare accuracy and training time with feature extraction and full fine-tuning.

### Exercise 2: Learning Rate Comparison
Run full fine-tuning with learning rates 1e-5, 2e-5, and 5e-5 (same epochs and data). Plot eval loss/accuracy and choose a best LR.

### Exercise 3: Different Task
Use the same pre-trained model for another task (e.g. another dataset from HuggingFace, or a 3-class classification). Keep feature extraction first, then try full fine-tuning.

### Exercise 4: Domain Data
If you have access to a small in-domain dataset (e.g. tweets, reviews, support tickets), run feature extraction and full fine-tuning and compare.

---

## Key Takeaways

1. **Transfer learning** reuses pre-trained LMs for new tasks with limited data.
2. **Feature extraction** (frozen backbone) is fast and stable; **fine-tuning** (unfreeze all) usually gives better accuracy.
3. Use a **small learning rate** when fine-tuning to avoid forgetting pre-trained knowledge.
4. **Domain adaptation** uses the same pipeline with in-domain data (pre-training or task data).
5. Always compare both strategies on your dataset and choose based on accuracy vs compute.

---

## Additional Resources

- [HuggingFace Transfer Learning](https://huggingface.co/docs/transformers/training)
- [ULMFiT](https://arxiv.org/abs/1801.06146), [BERT](https://arxiv.org/abs/1810.04805)

---

## Next Lesson

**L17: Fine-tuning Techniques** – Full fine-tuning, training loops, loss functions, optimization, and hyperparameter tuning for classification and generation.

---